In [104]:
import pandas as pd

Movie_path = r"C:\Users\adria\Desktop\Python\Machine-learning\labb1\ml-latest\movies.csv"
Rating_path = r"C:\Users\adria\Desktop\Python\Machine-learning\labb1\ml-latest\ratings.csv"
tags_path = r"C:\Users\adria\Desktop\Python\Machine-learning\labb1\ml-latest\tags.csv"

movieDF = pd.read_csv(Movie_path)
ratingsDF = pd.read_csv(Rating_path)
tagsDF = pd.read_csv(tags_path)

ratingsDF.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


In [105]:
movieDF.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [106]:
tagsDF.head()

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746


In [107]:
shapes = [f"Movie set shape: {movieDF.shape}", 
          f"Ratings set shape: {ratingsDF.shape}",
          f"Tags set shape: {tagsDF.shape}"]
for r in shapes:
    print(r)

Movie set shape: (86537, 3)
Ratings set shape: (33832162, 4)
Tags set shape: (2328315, 4)


In [108]:
movieDF.info()

<class 'pandas.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  86537 non-null  int64
 1   title    86537 non-null  str  
 2   genres   86537 non-null  str  
dtypes: int64(1), str(2)
memory usage: 2.0 MB


In [109]:
ratingsDF.info() 

<class 'pandas.DataFrame'>
RangeIndex: 33832162 entries, 0 to 33832161
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 1.0 GB


In [110]:
print(tagsDF.info())
print("unique tags:", len(tagsDF["tag"].unique()))

<class 'pandas.DataFrame'>
RangeIndex: 2328315 entries, 0 to 2328314
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   userId     int64
 1   movieId    int64
 2   tag        str  
 3   timestamp  int64
dtypes: int64(3), str(1)
memory usage: 71.1 MB
None
unique tags: 153950


In [111]:
movieDF["year"] = movieDF["title"].str.extract(r"\((\d{4})\)")
movieDF["title"] = movieDF["title"].str.replace(r"\((\d{4})\)", "", regex=True)

movieDF.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [112]:
tagsDF[tagsDF["tag"].isna()].head()

,userId,movieId,tag,timestamp
530012,104413,123,NaN,1199450867
530013,104413,346,NaN,1199451946
530017,104413,1184,NaN,1199452261
530024,104413,1785,NaN,1199452006
530025,104413,2194,NaN,1199450677


In [113]:
tags_grouped = tagsDF.groupby("movieId")["tag"].apply(lambda x: " ".join(x.dropna().astype(str))).reset_index()

tags_grouped.head()

,movieId,tag
0,1,animation friendship toys animation Disney Pix...
1,2,animals based on a book fantasy magic board ga...
2,3,sequel moldy old old age old men wedding old p...
3,4,characters chick flick girl movie characters c...
4,5,family pregnancy wedding 4th wall aging baby d...


In [124]:
movieDF["title"] = movieDF["title"].str.strip()

movieDF["title"][0]

'Toy Story'

In [125]:
movieTags = movieDF.merge(tags_grouped, on="movieId", how="left")

movieTags["tag"] = movieTags["tag"].fillna("")
movieTags["year"] = movieTags["year"].fillna("")

movieTags["text"] = (movieTags["title"].astype(str) + " " + 
                     movieTags["year"].astype(str) + " " +
                     movieTags["genres"].astype(str) + " " +
                     movieTags["tag"].astype(str))

movieTags["text"] = (movieTags["text"].str.lower().str.replace("|", " ", regex=False))

movieTags["text"][0]

"toy story 1995 adventure animation children comedy fantasy animation friendship toys animation disney pixar toys cgi classic disney pixar lots of heart tom hanks clever funny pixar witty animation comedy kids pixar tom hanks toy story toys animated fun jealousy funny witty adventure animation comedy family fantasy john lasseter usa family film friendship toys disney dolls national film registry buddy movie tom hanks witty pixar adventure clever enemies become friends family family cartoon family relationships feel-good first of series friends friendship fun fun family movie heroic mission humorous jealousy kids kids movie loyal friend loyalty redemption reflection rescue mission rivalry selflessness teamwork unlikely friendships unusual friendship witty os dois viram adventure animated animation cgi comedy disney family fantasy friendship imdb top 250 pixar tom hanks witty classic disney friendship pixar accepting reality emotional friendship funny soundtrack humorous pixar tumey's to

In [126]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df= 2
)

tfidf_matrix = vectorizer.fit_transform(movieTags["text"])

tfidf_matrix.shape

(86537, 300620)

In [127]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_similar_movies(movie_index, top_n = 20):
    movie_vector = tfidf_matrix[movie_index]
    similarities = cosine_similarity(movie_vector, tfidf_matrix).flatten()
    similar_indices = similarities.argsort()[::-1][1:top_n+1]
    return similar_indices, similarities[similar_indices]

In [309]:
def get_movie_index(movie_title):
    matches = movieDF.index[movieDF["title"].str.lower() == movie_title.lower()].tolist()

    if not matches:
        raise ValueError(f"The movie title: {movie_title} was not found")
    return matches[0]
    

def get_movie_titles(movie_indices):
    return movieDF.iloc[movie_indices]["title"].tolist()

def rertieval(movie_title, top_n=200):
    movie_indx = get_movie_index(movie_title)

    movie_vector = tfidf_matrix[movie_indx]
    similarities = cosine_similarity(movie_vector, tfidf_matrix).flatten()
    simMovieIndx = similarities.argsort()[::-1][1:top_n+1]

    simMovies = get_movie_titles(simMovieIndx)
    return simMovies, simMovieIndx

inputMovieTitle = "joe somebody"

candidateTitles, candidatesIndx = rertieval(inputMovieTitle)

for movie in candidateTitles[:10]:
    print(movie)


Who Is Cletis Tout?
Tim Allen: Men Are Pigs
Hemel
Slogans
Aamdani Atthanni Kharcha Rupaiya
Memórias Póstumas
10 Attitudes
Kwik Stop
Santa Clause, The
Stakaman!


In [310]:
def get_ratings(movies_byIndex, input_title):

    candidates_Id = movieTags.iloc[movies_byIndex]["movieId"]
    inputMovie_id = movieTags.loc[movieTags["title"].str.lower() == input_title.lower(), "movieId"].iloc[0]
    movies_to_keep = candidates_Id.tolist() + [inputMovie_id]
    candidateRatings = ratingsDF[ratingsDF["movieId"].isin(movies_to_keep)]
    candidateRatings.drop("timestamp", axis=1, inplace=True)
    
    return candidateRatings

In [311]:


candidates_Id = movieTags.iloc[candidatesIndx]["movieId"]

inputMovie_id = movieTags.loc[movieTags["title"].str.lower() == inputMovieTitle.lower(), "movieId"].iloc[0]

movies_to_keep = candidates_Id.tolist() + [inputMovie_id]

candidateRatings = ratingsDF[ratingsDF["movieId"].isin(movies_to_keep)]

candidateRatings.drop("timestamp", axis=1, inplace=True)
candidateRatings.head()

,userId,movieId,rating
116,2,317,3.0
374,7,1485,3.0
1266,21,317,3.0
2018,24,317,4.0
2150,24,1485,4.5


In [312]:
from scipy.sparse import csr_matrix

rows = candidateRatings["userId"].astype("category").cat.codes
cols = candidateRatings["movieId"].astype("category").cat.codes
data = candidateRatings["rating"]

rating_matrix = csr_matrix((data, (rows, cols)))

rating_matrix.shape

(46300, 196)

In [313]:
def build_sparse_matrix(ratings):
    rows = ratings["userId"].astype("category").cat.codes
    cols = ratings["movieId"].astype("category").cat.codes
    data = ratings["rating"]

    return csr_matrix((data, (rows, cols)))



In [314]:
movie_similarity = cosine_similarity(rating_matrix.T)

movie_similarity.shape

(196, 196)

In [315]:
movie_similarity = cosine_similarity(rating_matrix.T)

movie_categories = candidateRatings["movieId"].astype("category")
movieId_lookup = movie_categories.cat.categories

if inputMovie_id not in movieId_lookup:
    print("The movie has no ratings, CF cannot be used")
else:
    print("CF can be used")

CF can be used


In [316]:
def get_top5_from_ratings(rating_matrix, ratings, inputMovie_id):
    movie_similarity = cosine_similarity(rating_matrix.T)

    movie_categories = ratings["movieId"].astype("category")
    movieId_lookup = movie_categories.cat.categories

    if inputMovie_id in movieId_lookup:
        input_index = list(movieId_lookup).index(inputMovie_id)

        similarities = movie_similarity[input_index]

    else:
        similarities = None

    sorted_sim_indx = np.argsort(similarities)[::-1]

    sim_movieIds = movieId_lookup[sorted_sim_indx]

    rec_Ids = sim_movieIds.drop(inputMovie_id)

    return movieDF.loc[movieDF["movieId"].isin(rec_Ids[:5]), "title"] #recommendations

In [317]:
if inputMovie_id in movieId_lookup:
    input_index = list(movieId_lookup).index(inputMovie_id)

    similarities = movie_similarity[input_index]

else:
    similarities = None

sorted_sim_indx = np.argsort(similarities)[::-1]

sim_movieIds = movieId_lookup[sorted_sim_indx]

rec_Ids = sim_movieIds.drop(inputMovie_id)

recommendations = movieDF.loc[movieDF["movieId"].isin(rec_Ids[:5]), "title"]

recommendations

4059                 See Spot Run
5200      Taking Care of Business
6436                    Curly Sue
8265    Christmas with the Kranks
9448              In Good Company
Name: title, dtype: str

In [318]:
def ranking(candidateIndex, inputMovieTitle):
    candRatings = get_ratings(candidateIndex, inputMovieTitle)
    rating_matrix = build_sparse_matrix(candRatings)

    inputMovie_id = movieTags.loc[movieTags["title"].str.lower() == inputMovieTitle.lower(), "movieId"].iloc[0]

    return get_top5_from_ratings(rating_matrix, candRatings, inputMovie_id)
    

In [354]:
def twostage_RetrievalRanking(movieTitle):

    if movieTitle[:3].lower() == "the":
        movieTitle = movieTitle[4:]+", the"

    candidatesIndx = rertieval(movieTitle, top_n=200)[1]

    return ranking(candidatesIndx, movieTitle)

In [389]:
twostage_RetrievalRanking("braveheart")

351            Forrest Gump
522        Schindler's List
582      Dances with Wolves
1939    Saving Private Ryan
3480              Gladiator
Name: title, dtype: str